In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp1d
from scipy.ndimage import label
import optuna
from collections import deque
from loader import * 
from scripts.processing import *
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import LeaveOneOut

# Objective Function

In [4]:
def multi_objective(trial, measured_vs, average_fxs, average_fys, des_predicted_v, durations):

    delay = trial.suggest_float('delay', 0.01, 0.07) #davor: 0.07
    

    K_fy = trial.suggest_float('K_fy', -15.141400596291194e-07, -5.141400596291194e-07)#K_fy = trial.suggest_float('K_fy', -0.004, 1)
    K_fx = trial.suggest_float('K_fx', -0.002, 0.2)
    Kp = trial.suggest_float('Kp', 0.001, 20)
    Kd = trial.suggest_float('Kd', 0, 5e-16)#trial.suggest_float('Kd', 0, 0.2)

    n_trials = len(measured_vs)
    all_errors = []

    for n in range(n_trials):

        measured_v = measured_vs[n]
        des_speed = des_predicted_v[n]
        duration = durations[n]

        delta_t = duration / (len(measured_v))
        delay_samples = round(delay / delta_t)

        fy = [0 if i < 0 else i for i in average_fys[n]]
        fx = [0 if y < 0 else x
              for x, y in zip(average_fxs[n], average_fys[n])]
       # fy = [y for y in average_fys[n]]
      #  fx = [y for y in average_fxs[n]]
        
        measured_v = [des_speed if y < 10 else v
             for v, y in zip(measured_vs[n], average_fys[n])]

        start_index = int(90 * len(measured_v) / 100)

        predicted_v = np.zeros(len(measured_v))

        speed = des_speed
        predicted_v[start_index] = speed

        fx_all = deque(maxlen=delay_samples)
        fy_all = deque(maxlen=delay_samples)

        fx_all.append(0)
        fy_all.append(0)

        e_all = deque(maxlen=3)
        e = 0
        e_all.append(e)
       
        

        for k in range(len(measured_v) - 1):

            i = (start_index + k) % len(measured_v)
            next_i = (i + 1) % len(measured_v)

            

            if fy[next_i] <10: #this must match average v above
                speed_next = des_speed

                dfxdt = (fx[i] - fx[i - 1]) / delta_t
                dfydt = (fy[i] - fy[i - 1]) / delta_t

                fx_all.append(dfxdt)
                fy_all.append(dfydt)
                e = des_speed - speed
                e_all.append(e)
       

            else:
                dfxdt = (fx[i] - fx[i - 1]) / delta_t
                dfydt = (fy[i] - fy[i - 1]) / delta_t

                fx_all.append(dfxdt)
                fy_all.append(dfydt)

                e = des_speed - speed
                e_all.append(e)

             
                dedt = (e_all[1] - e_all[0]) / delta_t

                grf_part = K_fx * fx_all[0] + K_fy * fy_all[0]
                pd_part = Kp * e_all[1] + Kd * dedt

                speed_next = speed + grf_part + pd_part


            predicted_v[next_i] = speed_next
            speed = speed_next

        MSE = np.square(np.subtract(measured_v, predicted_v)).mean()
        all_errors.append(MSE)

    return np.mean(all_errors)

# Load data

In [5]:
exp_data_folder = Path("./data")
dataset = DatasetGrail(exp_data_folder, "./config.xlsx")
dataset = dataset.get_subset(
    task=["walking"], 
)
index = dataset.create_index()

## Initialization

In [6]:
x0 = {'delay': 0.05052497232666044,
 'K_fy': -9.141400596291194e-07,
 'K_fx': 4.3300646115648653e-06,
 'Kp': 0.6309665948387769,
 'Kd': 2.1007497833641600261e-17}

In [7]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Fit model on all available data

In [8]:
## from pathlib import Path
import numpy as np
import pandas as pd
import optuna

seeds = [0,1,100]
for seed in seeds: 
   


    print(seed)
    trainset = dataset
   

    vs, fys, fxs, durs, des_vs = [], [], [], [], []

    for p in trainset:
        cycles, dur = get_data(p)

        vs.append(cycles.speedL)
        fys.append(cycles.FyL)
        fxs.append(cycles.FxL)
        durs.append(dur)
        des_vs.append(p.index.speed[0])

    func_multi = lambda trial: multi_objective(
         trial, vs, fxs, fys, des_vs, durs
        )

    sampler = optuna.samplers.CmaEsSampler(x0=x0, seed=seed)
    study_multi = optuna.create_study(sampler=sampler)
    study_multi.optimize(func_multi, n_trials=4000)

    print(np.sqrt(study_multi.best_value))
    print(study_multi.best_params)
    
    
    

0
0.0028605810505755334
{'delay': 0.05188335825862577, 'K_fy': -9.263142150735485e-07, 'K_fx': 4.143226567605244e-06, 'Kp': 0.6539355723181786, 'Kd': 1.2842889273485065e-16}
1
0.0028854509253081234
{'delay': 0.05319509782619334, 'K_fy': -9.627210024383991e-07, 'K_fx': 4.284708410923037e-06, 'Kp': 0.6812029760510192, 'Kd': 1.7602227025006793e-16}
100
0.0028605810505755594
{'delay': 0.051928518676221266, 'K_fy': -9.263142205844286e-07, 'K_fx': 4.14322656464754e-06, 'Kp': 0.6539355700317108, 'Kd': 2.622079810108282e-16}
